# 01 - A Ground-Truth Causal Network

**Question:** Can our localization method recover a mechanism when we know the answer?

**Hypothesis:** `source` and `causal` form the true route. `nuisance` should have no causal effect.

**Expected compute:** CPU, seconds.

**What this establishes:** whether the implementation can recover a simple known causal structure.

**What it does not establish:** that the same method is complete or faithful on large real models.


In [ ]:
from neuros_mechint.benchmarks import GroundTruthCausalMLP, make_ground_truth_pair

model = GroundTruthCausalMLP().eval()
pair = make_ground_truth_pair()
print(model)
print("clean output:", model(pair.clean).item())
print("corrupted output:", model(pair.corrupted).item())


In [ ]:
from neuros_mechint import (
    ComponentRef, CounterfactualPair, MechanisticExperiment,
    OutputMetric, PatchIntervention, PyTorchAdapter,
)

experiment = MechanisticExperiment(
    adapter=PyTorchAdapter(model),
    pair=CounterfactualPair(pair.clean, pair.corrupted),
    metric=OutputMetric(lambda output: output.mean(), name="mean_output"),
    experiment_name="ground_truth_localization",
    model_id="GroundTruthCausalMLP",
    seed=42,
)

result = experiment.run([
    PatchIntervention(ComponentRef("source")),
    PatchIntervention(ComponentRef("causal")),
    PatchIntervention(ComponentRef("nuisance")),
])

for effect in result.effects:
    print(effect.component, "effect=", effect.effect, "recovered=", effect.recovered_fraction)


## Falsification check

The nuisance route is a negative control. If it receives a large effect, the method or experiment design is broken.

A useful interpretability toolkit should contain many fixtures like this, not only demos on systems whose true mechanism is unknown.
